# Population Screening & Exclusion Checks

This notebook checks a few candidate populations (cold-start transactions, amount/distance tails, start/end-of-window rows, missing/invalid values) to decide whether any should be excluded from training, and documents the reasoning for each.


## Setup


In [1]:
cd ../

/Users/ann/Documents/Zempler/fraud_datasets


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import src.config as config

pd.options.display.max_columns = 30

df_train = pd.read_csv(config.TRAIN_RAW_PATH, index_col=0)
TARGET = "is_fraud"


In [3]:
def fraud_summary(mask, label):
    """Summarize population size, share of dataset, and fraud rate for a boolean mask."""
    n = mask.sum()
    if n == 0:
        return {
            "group": label,
            "n": 0,
            "pct_population": 0,
            "fraud_rate": np.nan,
            "frauds": 0,
        }

    frauds = df_train.loc[mask, TARGET].sum()

    return {
        "group": label,
        "n": n,
        "pct_population": n / len(df_train),
        "fraud_rate": df_train.loc[mask, TARGET].mean(),
        "frauds": frauds,
    }


## Cold-start transactions

Checking whether a card's earliest transactions (1st, first 3, first 5) look different from the rest of the population in terms of fraud rate.


In [4]:
# Make sure transactions are ordered chronologically
df_check = df_train.sort_values(["cc_num", "trans_date_trans_time"]).copy()

df_check["card_txn_number"] = df_check.groupby("cc_num").cumcount() + 1

cold_start = pd.DataFrame([
    fraud_summary(df_check["card_txn_number"] == 1, "1st transaction"),
    fraud_summary(df_check["card_txn_number"] <= 3, "First 3 transactions"),
    fraud_summary(df_check["card_txn_number"] <= 5, "First 5 transactions"),
])

cold_start


,group,n,pct_population,fraud_rate,frauds
0,1st transaction,983,0.000758,0.076297,75
1,First 3 transactions,2949,0.002274,0.077314,228
2,First 5 transactions,4915,0.003790,0.078739,387


## Amount outliers

Checking fraud rate in the extreme tail of transaction amount.


In [5]:
for q in [0.99, 0.995, 0.999, 0.9999]:
    threshold = df_train["amt"].quantile(q)

    mask = df_train["amt"] >= threshold

    print(
        f"Amount >= {q:.3%} quantile "
        f"(£{threshold:,.2f}): "
        f"{mask.sum():,} rows, "
        f"{mask.mean():.2%} of population, "
        f"fraud rate = {df_train.loc[mask, TARGET].mean():.3%}"
    )


Amount >= 99.000% quantile (£545.99): 12,967 rows, 1.00% of population, fraud rate = 27.763%
Amount >= 99.500% quantile (£844.22): 6,484 rows, 0.50% of population, fraud rate = 36.952%
Amount >= 99.900% quantile (£1,499.25): 1,297 rows, 0.10% of population, fraud rate = 0.000%
Amount >= 99.990% quantile (£5,132.94): 130 rows, 0.01% of population, fraud rate = 0.000%


In [6]:
df_train["amt"].max()


28948.9

## Distance outliers

Distance between customer and merchant, computed via the haversine formula.


In [7]:
from math import radians, sin, cos, sqrt, atan2

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(
        np.radians, [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    return 6371 * 2 * np.arcsin(np.sqrt(a))


df_train["cust_merch_dist_km"] = haversine_km(
    df_train["lat"],
    df_train["long"],
    df_train["merch_lat"],
    df_train["merch_long"],
)


In [8]:
for q in [0.99, 0.995, 0.999]:
    threshold = df_train["cust_merch_dist_km"].quantile(q)
    mask = df_train["cust_merch_dist_km"] >= threshold

    print(
        f"Distance >= {q:.3%} quantile "
        f"({threshold:,.1f} km): "
        f"{mask.sum():,} rows, "
        f"{mask.mean():.2%} of population, "
        f"fraud rate = {df_train.loc[mask, TARGET].mean():.3%}"
    )


Distance >= 99.000% quantile (132.1 km): 12,967 rows, 1.00% of population, fraud rate = 0.578%
Distance >= 99.500% quantile (135.1 km): 6,484 rows, 0.50% of population, fraud rate = 0.679%
Distance >= 99.900% quantile (140.0 km): 1,297 rows, 0.10% of population, fraud rate = 0.617%


## Start / end of the time window

Checking whether the edges of the dataset's date range look different from the middle.


In [9]:
df_train["trans_date_trans_time"] = pd.to_datetime(
    df_train["trans_date_trans_time"]
)

print("Start:", df_train["trans_date_trans_time"].min())
print("End:  ", df_train["trans_date_trans_time"].max())


Start: 2019-01-01 00:00:18
End:   2020-06-21 12:13:37


In [10]:
start = df_train["trans_date_trans_time"].min()
end = df_train["trans_date_trans_time"].max()

for days in [1, 3, 7, 14]:
    start_mask = df_train["trans_date_trans_time"] < start + pd.Timedelta(days=days)
    end_mask = df_train["trans_date_trans_time"] >= end - pd.Timedelta(days=days)

    print(f"\nFirst {days} days:")
    print(f"  n = {start_mask.sum():,}")
    print(f"  fraud rate = {df_train.loc[start_mask, TARGET].mean():.3%}")

    print(f"Last {days} days:")
    print(f"  n = {end_mask.sum():,}")
    print(f"  fraud rate = {df_train.loc[end_mask, TARGET].mean():.3%}")



First 1 days:
  n = 2,414
  fraud rate = 0.000%
Last 1 days:
  n = 2,776
  fraud rate = 0.829%

First 3 days:
  n = 4,768
  fraud rate = 0.398%
Last 3 days:
  n = 7,364
  fraud rate = 1.127%

First 7 days:
  n = 12,106
  fraud rate = 0.421%
Last 7 days:
  n = 19,971
  fraud rate = 0.901%

First 14 days:
  n = 24,144
  fraud rate = 0.866%
Last 14 days:
  n = 39,851
  fraud rate = 0.662%


## Missing values


In [11]:
missing = (
    df_train.isna()
    .agg(["sum", "mean"])
    .T
    .rename(columns={"sum": "n_missing", "mean": "pct_missing"})
    .sort_values("n_missing", ascending=False)
)

missing[missing["n_missing"] > 0]


,n_missing,pct_missing


## Sanity checks: negative, zero, and infinite values


In [12]:
numeric_cols = df_train.select_dtypes(include="number").columns

sanity = pd.DataFrame({
    "n_negative": (df_train[numeric_cols] < 0).sum(),
    "n_zero": (df_train[numeric_cols] == 0).sum(),
    "n_inf": np.isinf(df_train[numeric_cols]).sum(),
})

sanity[(sanity != 0).any(axis=1)]


,n_negative,n_zero,n_inf
long,1296675,0,0
merch_long,1296675,0,0
is_fraud,0,1289169,0


## Summary screening table


In [13]:
screening = pd.DataFrame([
    fraud_summary(df_check["card_txn_number"] == 1, "First transaction"),
    fraud_summary(df_check["card_txn_number"] <= 3, "First 3 transactions"),
    fraud_summary(df_train["amt"] >= df_train["amt"].quantile(.99), "Top 1% amount"),
    fraud_summary(
        df_train["cust_merch_dist_km"] >= df_train["cust_merch_dist_km"].quantile(.99),
        "Top 1% distance"
    ),
])

screening


,group,n,pct_population,fraud_rate,frauds
0,First transaction,983,0.000758,0.076297,75
1,First 3 transactions,2949,0.002274,0.077314,228
2,Top 1% amount,12967,0.010000,0.277628,3600
3,Top 1% distance,12967,0.010000,0.005784,75
